In [ ]:
"""
============================================================================
ScienceGPT - Inference and Statistics Script
Load trained model, run inference, calculate training statistics
============================================================================
"""

import torch
import torch.nn as nn
from torch.nn import functional as F
import json
import glob
import os
from pathlib import Path
import numpy as np
from datetime import datetime, timedelta

# MODEL ARCHITECTURE

class Head(nn.Module):
    def __init__(self, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        tril_def = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer('tril', tril_def)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, E = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, n_embd, block_size, dropout)
                                    for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size, n_embd, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTModel(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.pos_emb_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_ffw_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.pos_emb_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_ffw_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        with torch.no_grad():
            for _ in range(max_new_tokens):
                idx_cond = idx if idx.size(1) <= self.block_size else idx[:, -self.block_size:]
                logits, _ = self(idx_cond)
                logits = logits[:, -1, :] / temperature

                if top_k is not None:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = -float('Inf')

                probs = F.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, num_samples=1)
                idx = torch.cat((idx, idx_next), dim=1)
        return idx


# LOAD MODEL AND VOCABULARY

def load_model_and_vocab(checkpoint_path='/content/drive/MyDrive/Character_Level/checkpoint_latest.pt',
                         vocab_path='/content/drive/MyDrive/Character_Level/vocab.json'):
    """Load trained model and vocabulary"""

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    # Check if checkpoint exists
    if not os.path.exists(checkpoint_path):
        print(f"\nCheckpoint not found: {checkpoint_path}")
        print("\nAvailable checkpoints:")
        checkpoints = sorted(glob.glob('./checkpoints/*.pt'))
        if checkpoints:
            for i, cp in enumerate(checkpoints, 1):
                print(f"  {i}. {cp}")
            return None, None, None, None
        else:
            print("  No checkpoints found in ./checkpoints/")
            return None, None, None, None

    # Load checkpoint
    print(f"\n Loading checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Extract config
    config = checkpoint['config']
    print(f"\n Checkpoint loaded from iteration {checkpoint['iter']}")

    # Load vocabulary
    if not os.path.exists(vocab_path):
        print(f"\n Vocabulary not found: {vocab_path}")
        return None, None, None, None

    with open(vocab_path, 'r', encoding='utf-8') as f:
        vocab_data = json.load(f)

    stoi = vocab_data['stoi']
    itos = {int(k): v for k, v in vocab_data['itos'].items()}
    vocab_size = vocab_data['vocab_size']

    print(f" Vocabulary loaded: {vocab_size} characters")

    # Create model
    print(f"\nLoading model...")
    model = GPTModel(
        vocab_size=config['vocab_size'],
        n_embd=config['n_embd'],
        n_head=config['n_head'],
        n_layer=config['n_layer'],
        block_size=config['block_size'],
        dropout=config['dropout']
    )

    # Load weights
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()

    n_params = sum(p.numel() for p in model.parameters())
    print(f" Model loaded: {n_params:,} parameters ({n_params/1e6:.1f}M)")

    return model, stoi, itos, checkpoint


# TRAINING STATISTICS


def analyze_training_stats(checkpoint, vocab_size):
    """Analyze and display training statistics"""

    print("\n" + "="*70)
    print("TRAINING STATISTICS")
    print("="*70)

    config = checkpoint['config']

    # Model architecture
    print("\n Model Architecture:")
    print(f"  Embedding dimension: {config['n_embd']}")
    print(f"  Number of layers: {config['n_layer']}")
    print(f"  Context length: {config['block_size']}")
    print(f"  Vocabulary size: {vocab_size}")
    print(f"  Dropout: {config['dropout']}")

    # Calculate parameters
    n_embd = config['n_embd']
    n_layer = config['n_layer']
    block_size = config['block_size']

    # Rough parameter count
    params_emb = vocab_size * n_embd + block_size * n_embd
    params_per_layer = (
        3 * n_embd * n_embd +  # Q, K, V
        n_embd * n_embd +      # Projection
        n_embd * 4 * n_embd +  # FFN up
        4 * n_embd * n_embd +  # FFN down
        4 * n_embd             # LayerNorms
    )
    params_total = params_emb + (params_per_layer * n_layer) + n_embd * vocab_size

    print(f"  Total parameters: ~{params_total:,} ({params_total/1e6:.1f}M)")

    # Training info
    print("\n Training Information:")
    print(f"  Iterations completed: {checkpoint['iter']:,}")
    print(f"  Best validation loss: {checkpoint.get('best_val_loss', 'N/A')}")

    # Estimate training details
    batch_size = 24  # From your config
    grad_accum = 8
    world_size = 2
    block_size = config['block_size']

    chars_per_iter = batch_size * block_size * grad_accum * world_size
    total_chars_processed = chars_per_iter * checkpoint['iter']

    print(f"  Characters per iteration: {chars_per_iter:,}")
    print(f"  Total characters processed: {total_chars_processed:,} ({total_chars_processed/1e9:.2f}B)")

    # Estimate from your dataset size
    dataset_size = 1.4e9  # Your reported size
    epochs_completed = total_chars_processed / dataset_size
    print(f"  Estimated epochs: {epochs_completed:.2f}")

    # Calculate training time estimate
    actual_iters = checkpoint['iter']
    if actual_iters >= 118000:
        estimated_hours = 29.3
    else:
        estimated_hours = (actual_iters / 118000) * 29.3

    print(f"\n Estimated Training Time:")
    print(f"  Approximate: {estimated_hours:.1f} hours ({estimated_hours/24:.2f} days)")

    # Cost estimate
    cost_per_hour = 7.58  # 2x H200 on RunPod
    estimated_cost = estimated_hours * cost_per_hour
    print(f"\n Estimated Cost:")
    print(f"  ~${estimated_cost:.2f} (at ${cost_per_hour}/hour for 2x H200)")

    print("\n" + "="*70)

# ============================================================================
# TEXT GENERATION
# ============================================================================

def generate_text(model, stoi, itos, device, prompt="", max_tokens=500,
                 temperature=0.75, top_k=160):
    """Generate text from prompt"""

    encode = lambda s: [stoi.get(c, 0) for c in s]
    decode = lambda l: ''.join([itos.get(i, '') for i in l])

    if prompt:
        encoded = encode(prompt)
        context = torch.tensor([encoded], dtype=torch.long, device=device)
    else:
        # Start with newline
        context = torch.zeros((1, 1), dtype=torch.long, device=device)

    generated = model.generate(
        context,
        max_new_tokens=max_tokens,
        temperature=temperature,
        top_k=top_k
    )

    result = decode(generated[0].tolist())
    return result


# INTERACTIVE Q&A


def interactive_qa(model, stoi, itos, device):
    """Interactive question-answering mode"""

    print("\n" + "="*70)
    print("="*70 + "\n")

    temperature = 0.75
    max_tokens = 500

    while True:
        try:
            prompt = input("\n Question: ").strip()

            if not prompt:
                continue

            if prompt.lower() in ['quit', 'exit', 'q', '/quit']:
                print("\n Goodbye!")
                break

            # Handle commands
            if prompt.startswith('/temp '):
                try:
                    temperature = float(prompt.split()[1])
                    temperature = max(0.1, min(2.0, temperature))
                    print(f"Temperature set to {temperature}")
                    continue
                except:
                    print("Invalid temperature. Use: /temp 0.8")
                    continue

            if prompt.startswith('/tokens '):
                try:
                    max_tokens = int(prompt.split()[1])
                    max_tokens = max(50, min(1000, max_tokens))
                    print(f" Max tokens set to {max_tokens}")
                    continue
                except:
                    print("Invalid token count. Use: /tokens 500")
                    continue

            # Generate response
            print("\n Answer:", end=' ', flush=True)
            response = generate_text(model, stoi, itos, device,
                                   prompt=prompt, max_tokens=max_tokens,
                                   temperature=temperature)
            print(response)

        except KeyboardInterrupt:
            print("\n\nGoodbye!")
            break
        except Exception as e:
            print(f"\nError: {e}")

# ============================================================================
# BATCH TESTING
# ============================================================================

def test_with_prompts(model, stoi, itos, device):
    """Test model with predefined scientific prompts"""

    print("\n" + "="*70)
    print("BATCH TESTING WITH SCIENTIFIC PROMPTS")
    print("="*70)

    test_prompts = [
        "What is the chemical composition of lead sulphate?",
        "Explain the discharge process in a battery.",
        "How does temperature affect battery performance?",
        "What causes sulphation in storage batteries?",
        "Describe the function of separators in a cell.",
        "What is the electrolyte made of?",
        "Explain specific gravity readings.",
        "How do you maintain a battery?",
    ]

    for i, prompt in enumerate(test_prompts, 1):
        print(f"\n{'='*70}")
        print(f"Test {i}/{len(test_prompts)}")
        print(f"{'='*70}")
        print(f"Prompt: {prompt}")
        print("-"*70)

        response = generate_text(model, stoi, itos, device,
                                prompt=prompt, max_tokens=300,
                                temperature=0.7, top_k=150)

        print(f"Response:\n{response}")

    print("\n" + "="*70)


# PERPLEXITY CALCULATION


def calculate_perplexity(model, data_sample, device):
    """Calculate perplexity on a data sample"""

    print("\n Calculating perplexity...")

    model.eval()
    total_loss = 0
    n_batches = 0

    # Take sample
    block_size = model.block_size
    sample_size = min(10000, len(data_sample) - block_size)

    with torch.no_grad():
        for i in range(0, sample_size, block_size):
            if i + block_size + 1 > len(data_sample):
                break

            x = data_sample[i:i+block_size].unsqueeze(0).to(device)
            y = data_sample[i+1:i+block_size+1].unsqueeze(0).to(device)

            _, loss = model(x, y)
            total_loss += loss.item()
            n_batches += 1

    avg_loss = total_loss / n_batches
    perplexity = torch.exp(torch.tensor(avg_loss)).item()

    print(f" Perplexity: {perplexity:.2f}")
    print(f"  (Lower is better, typical range: 10-100)")

    return perplexity

# ============================================================================
# MAIN FUNCTION
# ============================================================================

def main():
    """Main inference and statistics function"""

    print("\n" + "="*70)
    print("SCIENCEGPT - INFERENCE & STATISTICS")
    print("="*70)

    # Load model
    model, stoi, itos, checkpoint = load_model_and_vocab()

    if model is None:
        print("\nFailed to load model. Please check checkpoint path.")
        return

    device = next(model.parameters()).device

    # Show statistics
    analyze_training_stats(checkpoint, len(stoi))

    # Menu
    while True:
        print("\n" + "="*70)
        print("MENU")
        print("="*70)
        print("1. Interactive Q&A")
        print("2. Batch testing with sample prompts")
        print("="*70)

        choice = input("\nSelect option (1-7): ").strip()

        if choice == '1':
            interactive_qa(model, stoi, itos, device)

        elif choice == '2':
            test_with_prompts(model, stoi, itos, device)

        elif choice == '3':
            prompt = input("\nEnter your prompt: ").strip()
            if prompt:
                print(f"\nPrompt: {prompt}")
                print("-"*70)
                response = generate_text(model, stoi, itos, device, prompt=prompt)
                print(response)

        elif choice == '4':
            print("\nGenerating from empty prompt (creative mode)...")
            print("-"*70)
            response = generate_text(model, stoi, itos, device, prompt="")
            print(response)

        elif choice == '5':
            analyze_training_stats(checkpoint, len(stoi))

        elif choice == '6':
            print("\n Perplexity calculation requires loading training data.")
            print("This may take a while. Continue? (y/n)")
            if input().lower() == 'y':
                # Try to load some data
                data_dir = './scientific_data'
                if os.path.exists(data_dir):
                    files = glob.glob(f'{data_dir}/*.txt')
                    if files:
                        encode = lambda s: [stoi.get(c, 0) for c in s]
                        with open(files[0], 'r', encoding='utf-8', errors='ignore') as f:
                            sample_text = f.read()[:50000]  # First 50k chars
                        sample_data = torch.tensor(encode(sample_text), dtype=torch.long)
                        calculate_perplexity(model, sample_data, device)
                    else:
                        print("No data files found")
                else:
                    print("Data directory not found")

        elif choice == '7' or choice.lower() in ['quit', 'exit', 'q']:
            print("\n Goodbye!")
            break

        else:
            print(" Invalid option. Please select 1-7.")

# ============================================================================
# ENTRY POINT
# ============================================================================

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\n Goodbye!")
    except Exception as e:
        print(f"\n Error: {e}")
        import traceback
        traceback.print_exc()




SCIENCEGPT - INFERENCE & STATISTICS
Using device: cuda

 Loading checkpoint: /content/drive/MyDrive/Character_Level/checkpoint_latest.pt

 Checkpoint loaded from iteration 14438
 Vocabulary loaded: 2473 characters

Loading model...
 Model loaded: 89,222,569 parameters (89.2M)

TRAINING STATISTICS

 Model Architecture:
  Embedding dimension: 768
  Number of layers: 12
  Context length: 512
  Vocabulary size: 2473
  Dropout: 0.1
  Total parameters: ~89,163,264 (89.2M)

 Training Information:
  Iterations completed: 14,438
  Best validation loss: 0.9247406125068665
  Characters per iteration: 196,608
  Total characters processed: 2,838,626,304 (2.84B)
  Estimated epochs: 2.03

 Estimated Training Time:
  Approximate: 3.6 hours (0.15 days)

 Estimated Cost:
  ~$27.17 (at $7.58/hour for 2x H200)


MENU
1. Interactive Q&A
2. Batch testing with sample prompts

Select option (1-7): 3

Enter your prompt: What is an electolyte

Prompt: What is an electolyte
-------------------------------------